# Independent quantitative evaluation

This notebook evaluates saved outputs with AdaFace ID-Source, AdaFace ID-GT, DEX Age MAE, and KID. AdaFace and DEX checkpoints must already exist locally; this API never downloads them. Every `target_path` must be a real longitudinal photograph of the same person as its source.

In [ ]:
from pathlib import Path

from src.quantitative_metrics import evaluate_aging_outputs, load_quantitative_metrics

## 1. Load frozen evaluators

Use the official AdaFace R100/WebFace12M checkpoint and official DEX Caffe model. KID is loaded lazily, but its standard torch-fidelity Inception file must also exist locally; evaluator downloads are disabled.

In [ ]:
EVALUATOR_ROOT = Path("/content/evaluation_models")
MANIFEST_PATH = Path("/content/aging_outputs/prediction_manifest.csv")
EVALUATION_DIR = Path("/content/aging_outputs/evaluation")

metrics_bundle = load_quantitative_metrics(
    adaface_repo_path=EVALUATOR_ROOT / "AdaFace",
    adaface_checkpoint_path=EVALUATOR_ROOT / "adaface_ir101_webface12m.ckpt",
    dex_prototxt_path=EVALUATOR_ROOT / "DEX/age.prototxt",
    dex_checkpoint_path=EVALUATOR_ROOT / "DEX/dex_chalearn_iccv2015.caffemodel",
    kid_inception_weights_path=EVALUATOR_ROOT / "KID/weights-inception-2015-12-05-6726825d.pth",
    device="auto",
    local_files_only=True,
)

## 2. Evaluate a prediction manifest

`prediction_manifest.csv` needs `sample_id`, `source_path`, `target_path`, `generated_path`, `source_age`, `target_age`, and `model_name`. Missing target images are allowed: ID-GT and KID remain unavailable only for those rows, while ID-Source and DEX still run.

In [ ]:
evaluation = evaluate_aging_outputs(
    MANIFEST_PATH,
    metrics_bundle=metrics_bundle,
    output_dir=EVALUATION_DIR,
    run_name="paper_baselines",
    batch_size=32,
    kid_subsets=100,
    kid_subset_size=None,  # min(50, N)
)
display(evaluation["summary"])
display(evaluation["per_sample"].head())
print(evaluation["paths"])